<a href="https://colab.research.google.com/github/brianhphillips/testrepo/blob/main/IMDb_Casting_Director_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
import pandas as pd

# Mount Google Drive
drive.mount('/content/drive')

# IMDb dataset path in Drive
imdb_path = "/content/drive/MyDrive/"

# Target casting directors
target_names = {"Kevin Reher", "Natalie Lyon", "Jamie Sparer Roberts"}

Mounted at /content/drive


In [3]:
# Step 1 — Load only nconsts for our people
name_file = imdb_path + "name.basics.tsv"
chunk_size = 500_000
casting_ids = set()

for chunk in pd.read_csv(
    name_file,
    sep="\t",
    usecols=["nconst", "primaryName"],
    dtype=str,
    chunksize=chunk_size
):
    match_rows = chunk[chunk["primaryName"].isin(target_names)]
    casting_ids.update(match_rows["nconst"].tolist())

print(f"Found {len(casting_ids)} casting director IDs.")

Found 3 casting director IDs.


In [4]:
# Step 2 — Find titles where these IDs were casting directors
principals_file = imdb_path + "title.principals.tsv"
casting_titles = []

for chunk in pd.read_csv(
    principals_file,
    sep="\t",
    usecols=["tconst", "nconst", "category"],
    dtype=str,
    chunksize=chunk_size
):
    match_rows = chunk[
        (chunk["nconst"].isin(casting_ids)) &
        (chunk["category"].str.contains("casting", case=False, na=False))
    ]
    casting_titles.extend(match_rows.to_dict("records"))

casting_df = pd.DataFrame(casting_titles)

print(f"Found {casting_df['tconst'].nunique()} unique titles for our casting directors.")

Found 109 unique titles for our casting directors.


In [5]:
# Step 3 — Filter for movies only
movies_file = imdb_path + "title.basics.tsv"
movie_tconsts = set()

for chunk in pd.read_csv(
    movies_file,
    sep="\t",
    usecols=["tconst", "titleType"],
    dtype=str,
    chunksize=chunk_size
):
    match_rows = chunk[chunk["titleType"] == "movie"]
    movie_tconsts.update(match_rows["tconst"].tolist())

casting_df = casting_df[casting_df["tconst"].isin(movie_tconsts)]

print(f"After filtering for movies: {casting_df['tconst'].nunique()} titles remain.")

After filtering for movies: 43 titles remain.


In [6]:
# Step 4 — Add ratings
ratings_file = imdb_path + "title.ratings.tsv"
ratings_list = []

for chunk in pd.read_csv(
    ratings_file,
    sep="\t",
    usecols=["tconst", "averageRating"],
    dtype={"tconst": str, "averageRating": float},
    chunksize=chunk_size
):
    match_rows = chunk[chunk["tconst"].isin(casting_df["tconst"])]
    ratings_list.append(match_rows)

ratings_df = pd.concat(ratings_list, ignore_index=True)

In [7]:
# Step 5 — Merge and calculate averages
merged = casting_df.merge(ratings_df, on="tconst")
merged = merged.merge(
    pd.DataFrame(list(casting_ids), columns=["nconst"])
    .merge(pd.read_csv(name_file, sep="\t", usecols=["nconst", "primaryName"], dtype=str),
           on="nconst", how="left"),
    on="nconst", how="left"
)

avg_ratings = merged.groupby("primaryName")["averageRating"].mean().reset_index()

print(avg_ratings)

            primaryName  averageRating
0  Jamie Sparer Roberts       6.942857
1           Kevin Reher       7.425926
2          Natalie Lyon       7.429167
